In [0]:
%sql
INSERT OVERWRITE TABLE wishlink_datazip.brandcollab_post_base (
with post as (
    select sp.*, bc.is_paid, bc.name as campaign_name, bc.final_price
    from business_intelligence.silver_post_data sp
    join wishlink_datazip.brandcollab_e2e bc on sp.post_id = bc.post_id
),
product_description_1 as (
select ap.post_id, ap.id as product_id, ap.name product_name, description, 
b.name as brand,
ap.brand as sub_brand,
case when lower(ap.brand)<>lower(ap.platform) then ap.platform
else ap.ecomm_brand_name end as sub_brand_enriched,
sum(total_clicks) clicks, sum(commissionable_gmv + non_commissionable_gmv) sale
from silver_tables.atg_product_enriched ap
join post sp on sp.post_id = ap.post_id
left join business_intelligence.silver_product_daily spd on ap.id = spd.product_id
left join oldmonkey_production.bronze_atg_brand b on ap.brand_obj_id = b.id
group by all
),
product_description as (
SELECT
    post_id,
    to_json(
        collect_list(
            struct(
                product_id,
                product_name,
                description,
                brand,
                sub_brand, 
                sub_brand_enriched,
                round(sale , 0) sale,
                clicks
            )
        )
    ) AS products_json,
    collect_set(brand) AS brands,
    collect_set(sub_brand) AS subbrands,
    sum(clicks) total_clicks, sum(sale) as total_sale
FROM product_description_1
GROUP BY post_id
)
  select sp.*,
    pd.products_json, pd.total_clicks, pd.total_sale, brands, subbrands,
    get_json_object(post_details, '$.creator_gender') AS creator_gender,
    get_json_object(post_details, '$.video_url') AS video_url,
    tagging_results,
    get_json_object(transcription_result, '$.language')  AS transcription_language,
    get_json_object(transcription_result, '$.duration')  AS transcription_duration,
    get_json_object(transcription_result, '$.text')      AS transcription_text,

    get_json_object(tagging_results, '$.transcript_tagging_result.language')            AS tagged_transcript_language,
    get_json_object(tagging_results, '$.transcript_tagging_result.valid_transcript')    AS valid_transcript,
    get_json_object(tagging_results, '$.transcript_tagging_result.relevant_transcript') AS relevant_transcript,
    get_json_object(tagging_results, '$.transcript_tagging_result.summary')             AS transcript_summary,

    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].overall_score')   AS frame_overall_score,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].description')     AS frame_description,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].text_overlay')    AS text_overlay,

    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].aesthetic_scores.background_cohesion.score')            AS background_score,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].aesthetic_scores.background_cohesion.explanation')      AS background_explanation,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].aesthetic_scores.composition_framing.score')            AS composition_score,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].aesthetic_scores.composition_framing.explanation')      AS composition_explanation,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].aesthetic_scores.lighting_exposure.score')              AS lighting_score,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].aesthetic_scores.lighting_exposure.explanation')        AS lighting_explanation,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].aesthetic_scores.overall_aesthetic_appeal.score')       AS aesthetic_appeal_score,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].aesthetic_scores.overall_aesthetic_appeal.explanation') AS aesthetic_appeal_explanation,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].aesthetic_scores.styling_expression.score')             AS styling_score,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].aesthetic_scores.styling_expression.explanation')       AS styling_explanation,

    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].content_scores.content_polish.score')                AS content_polish_score,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].content_scores.content_polish.explanation')          AS content_polish_explanation,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].content_scores.contextual_relevance.score')          AS contextual_relevance_score,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].content_scores.contextual_relevance.explanation')    AS contextual_relevance_explanation,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].content_scores.marketing_fit.score')                 AS marketing_fit_score,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].content_scores.marketing_fit.explanation')           AS marketing_fit_explanation,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].content_scores.product_visibility.score')            AS product_visibility_score,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].content_scores.product_visibility.explanation')      AS product_visibility_explanation,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].content_scores.purpose_storytelling.score')          AS storytelling_score,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].content_scores.purpose_storytelling.explanation')    AS storytelling_explanation,

    get_json_object(tagging_results, '$.frame_tagging_result.response.summary.overall_score')       AS summary_overall_score,
    get_json_object(tagging_results, '$.frame_tagging_result.response.summary.content_category')    AS content_category,
    get_json_object(tagging_results, '$.frame_tagging_result.response.summary.content_subcategory') AS content_subcategory,
    get_json_object(tagging_results, '$.frame_tagging_result.response.summary.content_audience')    AS content_audience,
    get_json_object(tagging_results, '$.frame_tagging_result.response.summary.strengths')           AS strengths,
    get_json_object(tagging_results, '$.frame_tagging_result.response.summary.improvements')        AS improvements,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].keywords')          AS frame_keywords,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].topics')            AS frame_topics,

    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].products[0].product_name')  AS product_name,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].products[0].description')   AS product_description,
    get_json_object(tagging_results, '$.frame_tagging_result.response.frames[0].products[0].brand')         AS product_brand

from post sp
left join oldmonkey_production.bronze_atlas_posts atp
on sp.post_id = atp.post_id
left join product_description pd on sp.post_id = pd.post_id
);